# Noisy labeled mixture --- how fragile is a simple iterated relabel/refit?

2D mixture + swapped labels + inspect trajectory.

In [3]:
import numpy as np
rng = np.random.default_rng(7)
mus = np.array([[-1.1, -0.4], [1.4, 0.9]])  # hidden truth
sig = 0.45
n_per = 420
flip_fracs = np.linspace(0.0, 0.42, 10)
center_errs = []
paths = []


In [4]:
def dataset_with_flip(p: float):
    X0 = rng.normal(mus[0], sig, size=(n_per, 2))
    X1 = rng.normal(mus[1], sig, size=(n_per, 2))
    X = np.vstack([X0, X1])
    z = np.array([0] * n_per + [1] * n_per)
    y = z.copy()
    flip_m = rng.random(X.shape[0]) < p
    y[flip_m] = 1 - y[flip_m]
    return X, y


# naive plug-in centers from noisy labels
def naive_means(X, y):
    m0 = X[y == 0]
    m1 = X[y == 1]
    if m0.shape[0] < 8 or m1.shape[0] < 8:
        return None
    return np.vstack([m0.mean(axis=0), m1.mean(axis=0)])


X_t, y_t = dataset_with_flip(0.22)
mhat = naive_means(X_t, y_t)
print("sanity flip rate", np.mean(y_t != np.concatenate([np.zeros(n_per), np.ones(n_per)])))
print("naive centers", mhat)


sanity flip rate 0.19047619047619047
naive centers [[-0.62932631 -0.24934445]
 [ 0.92629327  0.63449065]]


In [5]:
# sweep corruption; same generator stream offset per p would confound—so fresh rng stream per iteration below
snap = []
for p in flip_fracs:
    rng_p = np.random.default_rng(hash((7, float(p))) % (2 ** 32))
    X0 = rng_p.normal(mus[0], sig, size=(n_per, 2))
    X1 = rng_p.normal(mus[1], sig, size=(n_per, 2))
    X = np.vstack([X0, X1])
    z = np.array([0] * n_per + [1] * n_per)
    y = z.copy()
    flip_m = rng_p.random(X.shape[0]) < p
    y[flip_m] = 1 - y[flip_m]
    mhat = naive_means(X, y)
    if mhat is None:
        center_errs.append(float("nan"))
        snap.append(None)
        continue
    err = np.linalg.norm(mhat[0] - mus[0]) + np.linalg.norm(mhat[1] - mus[1])
    center_errs.append(err)
    snap.append(mhat.copy())
center_errs = np.array(center_errs)
print("p grid", flip_fracs)
print("L1+L2 centroid error trace", np.round(center_errs, 3))


p grid [0.         0.04666667 0.09333333 0.14       0.18666667 0.23333333
 0.28       0.32666667 0.37333333 0.42      ]
L1+L2 centroid error trace [0.049 0.216 0.553 0.808 1.066 1.325 1.458 1.9   2.227 2.41 ]


In [6]:
# one refinement pass: Lloyd with hard assignment from iterated centers starting at naive_hat
rng_r = np.random.default_rng(11)
p_focus = float(flip_fracs[np.nanargmax(np.where(np.isfinite(center_errs), center_errs, -np.inf))])
X0 = rng_r.normal(mus[0], sig, size=(n_per, 2))
X1 = rng_r.normal(mus[1], sig, size=(n_per, 2))
X = np.vstack([X0, X1])
z = np.array([0] * n_per + [1] * n_per)
y = z.copy()
flip_m = rng_r.random(X.shape[0]) < p_focus
y[flip_m] = 1 - y[flip_m]
m = naive_means(X, y)
if m is None:
    traj = []
else:
    c = m.copy()
    traj = []
    for _ in range(14):
        d0 = np.linalg.norm(X - c[0], axis=1)
        d1 = np.linalg.norm(X - c[1], axis=1)
        assign = (d1 < d0).astype(int)
        c = np.array([X[assign == 0].mean(axis=0), X[assign == 1].mean(axis=0)])
        traj.append(np.linalg.norm(c[0] - mus[0]) + np.linalg.norm(c[1] - mus[1]))
print("worst-ish p_focus", round(p_focus, 3))
print("Lloyd traj", np.round(np.array(traj), 3))
improve = traj[0] - traj[-1] if traj else float("nan")
print("improvement naive->last", round(improve, 4))
import warnings
warnings.warn("if traj flat, overlapping components make label noise dominate", stacklevel=1)


worst-ish p_focus 0.42
Lloyd traj [0.037 0.037 0.037 0.037 0.037 0.037 0.037 0.037 0.037 0.037 0.037 0.037
 0.037 0.037]
improvement naive->last 0.0


C:\Users\goela\AppData\Local\Temp\ipykernel_20468\2227135417.py:28: UserWarning: if traj flat, overlapping components make label noise dominate
  warnings.warn("if traj flat, overlapping components make label noise dominate", stacklevel=1)


In [7]:
import matplotlib.pyplot as plt
plt.figure(figsize=(4.8, 2.9))
good = np.isfinite(center_errs)
plt.plot(flip_fracs[good], center_errs[good], "o-")
plt.xlabel("label flip probability")
plt.ylabel("||hat_mu-mu||_combined")
plt.tight_layout()
plt.savefig("examples/_synth_noisy_curve.png", dpi=120)
plt.close()
print("wrote examples/_synth_noisy_curve.png")


wrote examples/_synth_noisy_curve.png
